# Week 2 — Advanced RAG: Indexing & Vector Optimization

> **Source notebook for** [`src/rag_engine/`](../src/rag_engine/).

## Learning objectives

1. Diagnose why naive single-chunk RAG fails on long technical documents.
2. Derive the geometry of unit-normalized embeddings and the equivalence of cosine, dot product, and Euclidean distance on the unit sphere.
3. Implement parent-child hierarchical chunking and sentence-window retrieval, and explain when each is preferred.
4. Walk through HNSW from the level-assignment probability to the greedy search loop, using the from-scratch implementation in `src/rag_engine/hnsw_native.py`.
5. Ingest a corpus into a Chroma vector store and verify retrieval against held-out queries.


## 1. Why naive RAG fails

The simplest RAG pipeline:

```
doc → split into 500-token chunks → embed → store
query → embed → top-k cosine → concat → LLM
```

Two failure modes dominate:

**(a) Embedding fidelity collapses on long chunks.** Sentence-Transformers (Reimers & Gurevych, 2019) were trained on input lengths of roughly 50–200 tokens. As chunk length grows, the embedding becomes an averaged "topic" vector that no longer distinguishes between specific claims within the chunk. Retrieval becomes coarse.

**(b) Context starves on short chunks.** A 100-token chunk may contain the exact claim that matches the query, but the LLM needs surrounding context (definitions, antecedents, the rest of the paragraph) to answer correctly. Hallucination shoots up.

These objectives — **embedding fidelity** and **contextual sufficiency** — pull in opposite directions. Hierarchical and sentence-window retrieval *decouple* them.


## 2. Geometric foundation

For unit-norm vectors $\mathbf{u}, \mathbf{v} \in \mathbb{R}^d$:

$$
\cos(\mathbf{u}, \mathbf{v}) = \mathbf{u}^\top \mathbf{v},
\qquad
\|\mathbf{u} - \mathbf{v}\|_2^2 = 2 - 2\,\mathbf{u}^\top \mathbf{v}.
$$

Therefore **cosine similarity, inner product, and squared Euclidean distance produce the same ranking** on the unit sphere. This is why `src/rag_engine/embeddings.py` normalizes outputs: it eliminates an entire class of bugs where a Chroma collection configured for cosine and a Milvus collection configured for L2 silently disagree.


In [ ]:
import numpy as np

def normalize(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)

rng = np.random.default_rng(0)
u, v = normalize(rng.standard_normal((2, 8)).astype(np.float32))

cos_uv = float(u @ v)
l2_sq  = float(np.sum((u - v) ** 2))
print(f"cos(u,v) = {cos_uv:.4f}")
print(f"||u-v||² = {l2_sq:.4f}   ≟   2 - 2·cos = {2 - 2*cos_uv:.4f}")


## 3. Hierarchical (parent-child) chunking

**Idea.** Split each document into large *parents* (~1024 tokens, paragraph-aligned) and split each parent into smaller *children* (~256 tokens, sentence-aligned). Index the children for retrieval precision; return the parent text to the LLM for context.

The implementation lives in [`src/rag_engine/chunking.py`](../src/rag_engine/chunking.py).


In [ ]:
from src.rag_engine.chunking import HierarchicalChunker

doc = '''Retrieval-Augmented Generation (RAG) couples a parametric language model
with a non-parametric retriever. The retriever indexes a corpus of text passages
and, at query time, returns the top-k passages by similarity to the query. These
passages are concatenated into the prompt for the generator.

A common failure mode is *chunk geometry mismatch*. Embedding models like SBERT
were trained on short inputs. Their vector quality degrades as input length grows.
Long chunks therefore embed poorly and retrieval recall suffers.

At the same time, short chunks deprive the generator of the surrounding context
it needs to answer. A two-sentence excerpt may match the query exactly and yet
omit the antecedent the LLM needs to interpret it. This produces fluent but
incorrect answers — the classic hallucination signature.

Hierarchical chunking decouples these objectives. Each document is split into
large *parent* chunks for context and smaller *child* chunks for retrieval.
At inference time the child hit triggers parent expansion: the LLM sees the full
parent text, while the retriever ranks against the focused child embedding.'''

chunker = HierarchicalChunker(parent_chars=600, child_chars=200, child_overlap_chars=40)
parents, children = chunker.chunk(doc, doc_metadata={"doc_id": "rag-101"})

print(f"{len(parents)} parents, {len(children)} children")
print("---")
print("First child:", children[0].text[:160], "...")
print("Parent it belongs to:", parents[0].chunk_id == children[0].parent_id)


**Why overlap.** Children carry a small overlap into the next chunk (`child_overlap_chars`). This mitigates the *boundary effect*: a query whose answer straddles a chunk boundary would otherwise miss both sides. Overlap costs storage and embeds more vectors, but in practice yields measurable recall gains on long technical text.


## 4. Sentence-window retrieval

**Idea.** Index *individual sentences* for maximum embedding precision. At retrieval time, expand the hit into a window of ±W neighbouring sentences before handing it to the LLM.

This is the dual strategy: extreme precision at index time, contextual expansion at retrieval time. It works exceptionally well for question-answering corpora where the *answer* is one sentence but the *context* is a paragraph.


In [ ]:
from src.rag_engine.chunking import SentenceWindowChunker

sw = SentenceWindowChunker(window=2)
sw_chunks = sw.chunk(doc, doc_metadata={"doc_id": "rag-101"})
print(f"{len(sw_chunks)} sentence chunks")
print("hit:", sw_chunks[5].text)
print("---")
print("expanded ±2:", sw.expand_window(sw_chunks[5], {"rag-101": sw_chunks}))


## 5. HNSW from first principles

For large corpora, exhaustive cosine search is $O(N)$ per query — fine for $N=10^3$, fatal at $N=10^7$. HNSW (Malkov & Yashunin, 2018) achieves expected $O(\log N)$ search.

### Level assignment

A new point is assigned a maximum layer

$$
\ell = \lfloor -\ln(u) \cdot m_L \rfloor, \quad u \sim U(0,1), \quad m_L = \frac{1}{\ln M}.
$$

This produces a geometric distribution over layers: roughly $N/M^\ell$ points reach layer $\ell$. The top layer behaves like a sparse small-world graph; the bottom is a dense kNN graph.


In [ ]:
import math, random
random.seed(0)

M = 16
m_L = 1.0 / math.log(M)
levels = [int(math.floor(-math.log(random.random() + 1e-12) * m_L)) for _ in range(10_000)]

from collections import Counter
counts = Counter(levels)
for layer in sorted(counts):
    bar = "█" * int(counts[layer] * 60 / 10_000)
    print(f"layer {layer:2d}: {counts[layer]:6d}  {bar}")


The exponential decay is exactly what makes greedy descent log-time: the top layers contain few points, so the search funnels rapidly toward the bottom layer where the answer lives.


### Greedy search

From the entry point at the top layer:

1. Compute the distance to all neighbors of the current node.
2. If any neighbor is closer to the query than the current node, jump to it.
3. Repeat until no neighbor is closer.
4. Drop to the next layer down and continue.
5. At layer 0, expand into a priority-queue search of size `ef` for the final top-k.


In [ ]:
from src.rag_engine.hnsw_native import HNSWIndex

# Build a small index and inspect its behavior.
rng = np.random.default_rng(1)
N, d = 1000, 64
vectors = normalize(rng.standard_normal((N, d)).astype(np.float32))

index = HNSWIndex(dim=d, M=16, ef_construction=100, ef_search=50, seed=42)
for i, v in enumerate(vectors):
    index.add(v, payload={"id": i})

print(f"max layer reached: {index._max_layer}")
print(f"entry point id: {index._entry_point}")

# Sanity check: search a known point. Should return that same point at distance ≈ 0.
hit = index.search(vectors[42], k=3)
for dist, payload in hit:
    print(f"  id={payload['id']:4d}  dist={dist:.6f}")


In [ ]:
# Compare against brute force on a held-out query.
query = normalize(rng.standard_normal(d).astype(np.float32))

# Brute force ground truth.
bf_dists = 1.0 - vectors @ query
bf_top = np.argsort(bf_dists)[:10]

# HNSW approximate top-10.
hnsw_top = [p["id"] for _, p in index.search(query, k=10)]

print("recall@10:", len(set(bf_top.tolist()) & set(hnsw_top)) / 10)


For uniformly distributed data, HNSW typically achieves $\geq 0.95$ recall@10 with `ef_search=50`. Tuning `ef_search` is the dominant lever: higher values trade query latency for recall.


## 6. End-to-end: ingest into Chroma

The production path uses Chroma's HNSW (a faster, C-backed implementation) behind the same `VectorStore` interface. The from-scratch index above is for understanding the mathematics; in code you'd use:


In [ ]:
from src.rag_engine.embeddings import SentenceEncoder
from src.rag_engine.pipeline import HierarchicalRAG
from src.rag_engine.vector_stores import build_store
import tempfile

persist = tempfile.mkdtemp()
rag = HierarchicalRAG(
    chunker=HierarchicalChunker(parent_chars=600, child_chars=200, child_overlap_chars=40),
    encoder=SentenceEncoder("BAAI/bge-small-en-v1.5"),
    child_store=build_store("chroma", collection="demo", persist_dir=persist),
    parent_lookup={},
)

# Ingest three short docs.
docs = [
    ("hyde",     "HyDE generates a hypothetical answer and embeds it for retrieval. The synthetic document is closer to real answer documents than the query is."),
    ("rerank",   "Cross-encoders score query-document pairs jointly. They are slower than bi-encoders but much more accurate, making them ideal for second-stage re-ranking."),
    ("chunking", "Hierarchical chunking splits documents into parent and child chunks. Children are indexed for retrieval precision; parents are returned for context."),
]
for doc_id, text in docs:
    rag.ingest_document(text, metadata={"doc_id": doc_id, "title": doc_id})

# Query it.
for q in ["how does re-ranking work?", "what is HyDE?", "parent-child chunks"]:
    hits = rag.retrieve(q, k=2)
    print(f"\nQ: {q}")
    for h in hits:
        print(f"  [{h.score:.3f}] {h.metadata.get('doc_id')}  {h.text[:80]}…")


## 7. Exercises

1. **Tune the chunk geometry.** On a held-out QA dataset, sweep `(parent_chars, child_chars)` over a grid. Plot recall@10 vs. mean LLM answer-correctness. Where is the Pareto frontier?
2. **Implement IVF-PQ.** HNSW is one ANN family; Inverted File + Product Quantization is another. Implement a toy IVF-PQ index and compare its memory footprint to HNSW on the same data.
3. **Sentence-window vs. hierarchical.** Pick a corpus where one strategy clearly wins. Articulate *why* — the answer should reference the typical query/answer length ratio in that corpus.


## 8. Take-aways

- The chunk geometry is the single most important hyperparameter in a RAG system. Hierarchical and sentence-window strategies *decouple* the embedding-fidelity / context-sufficiency tradeoff.
- **Unit-normalize all embeddings**. Cosine, dot, and Euclidean become equivalent rankings, and an entire class of backend-mismatch bugs disappears.
- HNSW's expected $O(\log N)$ search rests on a geometric layer-occupancy distribution, achieved via a simple level-sampling formula.
- The `VectorStore` Protocol means the choice of backend is a config option, not an architectural commitment.

➡ Next week we attack the *other* axis: query transformation (HyDE) and cross-encoder re-ranking.
